<a href="https://colab.research.google.com/github/Akhila24535/E-commerce-web-application/blob/main/E_commerce_web_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# INSTALL REQUIRED PACKAGES
!pip install flask flask_sqlalchemy pyngrok -q

from flask import Flask, request, jsonify
from flask_sqlalchemy import SQLAlchemy
from pyngrok import ngrok
import threading

# CREATE APP
app = Flask(__name__)
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///ecommerce.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)

# DATABASE TABLES
class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(100), unique=True, nullable=False)
    password = db.Column(db.String(100), nullable=False)
    role = db.Column(db.String(20), default='user')

class Product(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    name = db.Column(db.String(100))
    price = db.Column(db.Float)
    stock = db.Column(db.Integer)

class Order(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(100))
    product_name = db.Column(db.String(100))
    quantity = db.Column(db.Integer)
    total_price = db.Column(db.Float)

with app.app_context():
    db.create_all()

    # SAMPLE PRODUCTS
    if Product.query.count() == 0:
        products = [
            Product(name="Laptop", price=55000, stock=10),
            Product(name="Smartphone", price=25000, stock=20),
            Product(name="Headphones", price=2000, stock=30),
            Product(name="Smart Watch", price=5000, stock=15)
        ]
        db.session.add_all(products)
        db.session.commit()

# HOME
@app.route('/')
def home():
    return """
    <h1>E-Commerce Web Application</h1>
    <p>Available APIs:</p>
    <ul>
        <li>POST /register</li>
        <li>POST /login</li>
        <li>GET /products</li>
        <li>POST /add_product</li>
        <li>POST /order</li>
        <li>GET /orders</li>
    </ul>
    """

# REGISTER
@app.route('/register', methods=['POST'])
def register():
    data = request.json

    if User.query.filter_by(username=data['username']).first():
        return jsonify({"message": "User already exists"})

    user = User(
        username=data['username'],
        password=data['password'],
        role=data.get('role', 'user')
    )

    db.session.add(user)
    db.session.commit()

    return jsonify({"message": "Registration Successful"})

# LOGIN
@app.route('/login', methods=['POST'])
def login():
    data = request.json

    user = User.query.filter_by(
        username=data['username'],
        password=data['password']
    ).first()

    if user:
        return jsonify({
            "message": "Login Successful",
            "role": user.role
        })

    return jsonify({"message": "Invalid Credentials"}), 401

# VIEW PRODUCTS
@app.route('/products', methods=['GET'])
def products():
    product_list = Product.query.all()

    result = [{
        "id": p.id,
        "name": p.name,
        "price": p.price,
        "stock": p.stock
    } for p in product_list]

    return jsonify(result)

# ADD PRODUCT (ADMIN)
@app.route('/add_product', methods=['POST'])
def add_product():
    data = request.json

    product = Product(
        name=data['name'],
        price=data['price'],
        stock=data['stock']
    )

    db.session.add(product)
    db.session.commit()

    return jsonify({"message": "Product Added Successfully"})

# PLACE ORDER
@app.route('/order', methods=['POST'])
def place_order():
    data = request.json

    product = Product.query.get(data['product_id'])

    if not product:
        return jsonify({"message": "Product Not Found"})

    if product.stock < data['quantity']:
        return jsonify({"message": "Not Enough Stock"})

    total = product.price * data['quantity']

    order = Order(
        username=data['username'],
        product_name=product.name,
        quantity=data['quantity'],
        total_price=total
    )

    product.stock -= data['quantity']

    db.session.add(order)
    db.session.commit()

    return jsonify({
        "message": "Order Placed Successfully",
        "total_price": total
    })

# VIEW ORDERS
@app.route('/orders', methods=['GET'])
def orders():
    order_list = Order.query.all()

    result = [{
        "id": o.id,
        "username": o.username,
        "product": o.product_name,
        "quantity": o.quantity,
        "total_price": o.total_price
    } for o in order_list]

    return jsonify(result)

# START SERVER
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5000)
).start()

ERROR:pyngrok.process.ngrok:t=2026-06-11T13:38:19+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-11T13:38:19+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.